# Logistic Regression (Softmax) from Scratch on PlantVillage

Bu bölümde:

1. preprocessed_plantvillage klasöründen train/val/test verilerini yükleyeceğiz.
2. Sıfırdan yazılmış Multiclass Logistic Regression (Softmax Regression) modelini tanımlayacağız.
3. Modeli eğitip train/validation/test accuracy hesaplayacağız.
4. Accuracy, Precision, Recall, F1 ve Confusion Matrix'i kütüphanesiz olarak hesaplayacağız.



## 🧩 Cell 1 – preprocessed veriyi yükleme (Code)
Bu hücrede .npy dosyalarını yüklüyoruz ve Logistic Regression sınıfımız listelerle çalıştığı için Python listesine çeviriyoruz.

In [1]:
import numpy as np

# Load preprocessed data
SAVE_DIR = "preprocessed_plantvillage"
X_train = np.load(f"{SAVE_DIR}/train_X.npy")
y_train = np.load(f"{SAVE_DIR}/train_y.npy")
X_val = np.load(f"{SAVE_DIR}/val_X.npy")
y_val = np.load(f"{SAVE_DIR}/val_y.npy")
X_test = np.load(f"{SAVE_DIR}/test_X.npy")
y_test = np.load(f"{SAVE_DIR}/test_y.npy")

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

num_features = X_train.shape[1]
num_classes = len(set(y_train.tolist()))
print("Number of features:", num_features)
print("Number of classes:", num_classes)

# Convert numpy arrays to Python lists (for our pure Python implementation)
X_train_list = X_train.tolist()
y_train_list = y_train.tolist()
X_val_list = X_val.tolist()
y_val_list = y_val.tolist()
X_test_list = X_test.tolist()
y_test_list = y_test.tolist()



Train shape: (12537, 1024) (12537,)
Validation shape: (2686, 1024) (2686,)
Test shape: (2687, 1024) (2687,)
Number of features: 1024
Number of classes: 38


- `.npy` dosyalarından `X_train`, `y_train`, `X_val`, `X_test` yükleniyor.
- `X` array’lerinin shape’lerini kontrol ediyoruz (örnek: `(N, 1024)`).
- Sınıf sayısını (`num_classes`) buluyoruz.
- Logistic regression implementasyonumuz Python listeleriyle çalıştığı için `.tolist()` ile listelere çeviriyoruz.



## 🧩 Cell 2 – Logistic Regression modelini tanımlama (Code)
Bu hücrede sıfırdan Logistic Regression (Softmax) sınıfını tanımlıyoruz. Bu, Task 2’de “kütüphanesiz reimplement” kısmın olacak.

In [2]:
import math
import random

def softmax(logits):
    """
    Compute softmax probabilities from raw logits.
    logits: list[float] of length C
    returns: list[float] of length C (sum to 1)
    """
    max_logit = max(logits)
    exps = [math.exp(z - max_logit) for z in logits]
    sum_exps = sum(exps)
    if sum_exps == 0.0:
        c = len(logits)
        return [1.0 / c for _ in range(c)]
    return [e / sum_exps for e in exps]

def cross_entropy_loss(probs, true_class):
    """
    Compute cross-entropy loss for one example.
    probs: list[float] of length C (softmax output)
    true_class: int (0..C-1)
    """
    eps = 1e-15
    p = probs[true_class]
    p = max(min(p, 1.0 - eps), eps)
    return -math.log(p)

class MulticlassLogisticRegression:
    """
    Simple multiclass logistic regression (softmax regression)
    implemented from scratch using only Python lists.
    """

    def __init__(self, num_features, num_classes, learning_rate=0.1):
        self.num_features = num_features
        self.num_classes = num_classes
        self.learning_rate = learning_rate
        # Initialize weights and biases with small random values
        # W shape: (num_features, num_classes)
        self.W = [
            [(random.random() - 0.5) * 0.01 for _ in range(num_classes)]
            for _ in range(num_features)
        ]
        # b shape: (num_classes,)
        self.b = [(random.random() - 0.5) * 0.01 for _ in range(num_classes)]

    def _compute_logits(self, x):
        """
        Compute logits (raw scores) for one example.
        x: list[float] of length num_features
        returns: list[float] of length num_classes
        """
        logits = [0.0 for _ in range(self.num_classes)]
        for k in range(self.num_classes):
            s = 0.0
            for j in range(self.num_features):
                s += x[j] * self.W[j][k]
            s += self.b[k]
            logits[k] = s
        return logits

    def predict_proba_one(self, x):
        """
        Predict class probabilities for a single example.
        """
        logits = self._compute_logits(x)
        probs = softmax(logits)
        return probs

    def predict_one(self, x):
        """
        Predict class index for a single example.
        """
        probs = self.predict_proba_one(x)
        best_class = 0
        best_prob = probs[0]
        for k in range(1, self.num_classes):
            if probs[k] > best_prob:
                best_prob = probs[k]
                best_class = k
        return best_class

    def predict(self, X):
        """
        Predict class indices for a list of examples.
        X: list of examples, each example is list[float]
        """
        return [self.predict_one(x) for x in X]

    def fit(self, X, y, num_epochs=10, shuffle=True, verbose=True,
            X_val=None, y_val=None):
        """
        Train the model using stochastic gradient descent.
        X: list of examples, each example is list[float] of length num_features
        y: list[int] of class indices (0..num_classes-1)
        num_epochs: number of passes over the training set
        X_val, y_val: optional validation set for monitoring
        """
        n_samples = len(X)

        for epoch in range(num_epochs):
            indices = list(range(n_samples))
            if shuffle:
                random.shuffle(indices)

            total_loss = 0.0
            correct = 0

            # Progress settings: show progress in ~10 steps (0%, 10%, 20%, ...)
            progress_step = max(1, n_samples // 10)

            for step, idx in enumerate(indices):
                x = X[idx]
                true_class = y[idx]

                # Forward pass
                logits = self._compute_logits(x)
                probs = softmax(logits)
                loss = cross_entropy_loss(probs, true_class)
                total_loss += loss

                # Accuracy tracking (train)
                pred_class = 0
                best_prob = probs[0]
                for k in range(1, self.num_classes):
                    if probs[k] > best_prob:
                        best_prob = probs[k]
                        pred_class = k
                if pred_class == true_class:
                    correct += 1

                # Backward pass + parameter update (SGD)
                for k in range(self.num_classes):
                    if k == true_class:
                        error_k = probs[k] - 1.0
                    else:
                        error_k = probs[k]

                    for j in range(self.num_features):
                        grad_w_jk = error_k * x[j]
                        self.W[j][k] -= self.learning_rate * grad_w_jk

                    grad_b_k = error_k
                    self.b[k] -= self.learning_rate * grad_b_k

                # Progress print for this epoch
                if verbose and (step + 1) % progress_step == 0:
                    pct = (step + 1) / n_samples * 100.0
                    print(
                        f"Epoch {epoch + 1}/{num_epochs} - "
                        f"{pct:5.1f}% completed",
                        end="\r",
                        flush=True
                    )

            # ---- Epoch finished: compute train stats ----
            avg_loss = total_loss / n_samples
            train_accuracy = correct / n_samples

            # ---- Optional: compute validation accuracy ----
            val_accuracy = None
            if X_val is not None and y_val is not None and len(X_val) > 0:
                # Simple accuracy on validation set
                correct_val = 0
                for x_val, y_true_val in zip(X_val, y_val):
                    y_pred_val = self.predict_one(x_val)
                    if y_pred_val == y_true_val:
                        correct_val += 1
                val_accuracy = correct_val / len(X_val)

            if verbose:
                if val_accuracy is not None:
                    print(
                        f"Epoch {epoch + 1}/{num_epochs} - "
                        f"loss: {avg_loss:.4f} - "
                        f"train_acc: {train_accuracy:.4f} - "
                        f"val_acc: {val_accuracy:.4f}          "
                    )
                else:
                    print(
                        f"Epoch {epoch + 1}/{num_epochs} - "
                        f"loss: {avg_loss:.4f} - "
                        f"train_acc: {train_accuracy:.4f}          "
                    )




Bu sınıf:
- `W` ve `b` parametrelerini küçük rastgele değerlerle başlatıyor.
- `softmax + cross-entropy` ile loss hesaplıyor.
- Gradient descent ile `W` ve `b`’yi güncelliyor.
- `fit()` içinde epoch’lar boyunca veri üzerinde dolaşıyor ve loss + accuracy yazdırıyor.
Bu tamamen senin Logistic Regression reimplementasyonun oluyor.

## 🧩 Cell 3 – Modeli eğitme (Code)


In [3]:
learning_rate = 0.01
num_epochs = 20

model = MulticlassLogisticRegression(
    num_features=num_features,
    num_classes=num_classes,
    learning_rate=learning_rate
)

model.fit(
    X_train_list,
    y_train_list,
    num_epochs=num_epochs,
    verbose=True,
    X_val=X_val_list,
    y_val=y_val_list
)


Epoch 1/20 - loss: 3.4732 - train_acc: 0.1162 - val_acc: 0.0823          
Epoch 2/20 - loss: 3.1089 - train_acc: 0.1803 - val_acc: 0.2208          
Epoch 3/20 - loss: 2.9756 - train_acc: 0.2139 - val_acc: 0.1880          
Epoch 4/20 - loss: 2.9123 - train_acc: 0.2259 - val_acc: 0.2290          
Epoch 5/20 - loss: 2.8288 - train_acc: 0.2525 - val_acc: 0.2141          
Epoch 6/20 - loss: 2.7898 - train_acc: 0.2634 - val_acc: 0.2420          
Epoch 7/20 - loss: 2.7363 - train_acc: 0.2673 - val_acc: 0.2401          
Epoch 8/20 - loss: 2.7061 - train_acc: 0.2767 - val_acc: 0.2714          
Epoch 9/20 - loss: 2.6693 - train_acc: 0.2850 - val_acc: 0.2014          
Epoch 10/20 - loss: 2.6494 - train_acc: 0.2797 - val_acc: 0.1813          
Epoch 11/20 - loss: 2.6122 - train_acc: 0.2915 - val_acc: 0.2383          
Epoch 12/20 - loss: 2.5906 - train_acc: 0.2982 - val_acc: 0.2684          
Epoch 13/20 - loss: 2.5674 - train_acc: 0.3084 - val_acc: 0.2766          
Epoch 14/20 - loss: 2.5511 - train

- İlk deneme için `learning_rate = 0.1`, `num_epochs = 10` ile başlayabilirsin.
- Eğitim yavaşsa epoch sayısını azaltabilir, hız iyi ise artırabilirsin.
- Çıktıda her epoch için `loss` ve `accuracy` göreceksin.



## 🧩 Cell 4 – Accuracy hesaplama (Code)


In [4]:
def accuracy_score(y_true, y_pred):
    """
    Compute simple accuracy = correct / total.
    """
    assert len(y_true) == len(y_pred)
    correct = sum(int(t == p) for t, p in zip(y_true, y_pred))
    return correct / len(y_true)

# Evaluate on train, val, test
y_train_pred = model.predict(X_train_list)
y_val_pred = model.predict(X_val_list)
y_test_pred = model.predict(X_test_list)

train_acc = accuracy_score(y_train_list, y_train_pred)
val_acc = accuracy_score(y_val_list, y_val_pred)
test_acc = accuracy_score(y_test_list, y_test_pred)

print(f"Train accuracy: {train_acc:.4f}")
print(f"Validation accuracy: {val_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")



Train accuracy: 0.4027
Validation accuracy: 0.3019
Test accuracy: 0.3104


- Basit accuracy: doğru tahmin sayısı / toplam örnek sayısı.
- Train / Validation / Test accuracy’lerini ayrı ayrı hesaplıyoruz.
- Task 2’de “deneyler” kısmında bu değerleri tabloya koyacaksın.



## 🧩 Cell 5 – Confusion Matrix, Precision, Recall, F1 (kütüphanesiz) (Code)
Şimdi metrikler için saf Python fonksiyonları:

In [5]:
def confusion_matrix(y_true, y_pred, num_classes):
    """
    Compute confusion matrix as a 2D list.
    Rows: true classes
    Columns: predicted classes
    """
    mat = [[0 for _ in range(num_classes)] for _ in range(num_classes)]
    for t, p in zip(y_true, y_pred):
        mat[t][p] += 1
    return mat

def precision_recall_f1_per_class(y_true, y_pred, num_classes):
    """
    Compute precision, recall, and F1 for each class.
    Returns:
        precisions, recalls, f1s: lists of length num_classes
    """
    cm = confusion_matrix(y_true, y_pred, num_classes)
    precisions = []
    recalls = []
    f1s = []
    for c in range(num_classes):
        tp = cm[c][c]
        fp = sum(cm[r][c] for r in range(num_classes) if r != c)
        fn = sum(cm[c][k] for k in range(num_classes) if k != c)
        # Precision = TP / (TP + FP)
        if tp + fp == 0:
            precision = 0.0
        else:
            precision = tp / (tp + fp)
        # Recall = TP / (TP + FN)
        if tp + fn == 0:
            recall = 0.0
        else:
            recall = tp / (tp + fn)
        # F1 = 2 * P * R / (P + R)
        if precision + recall == 0:
            f1 = 0.0
        else:
            f1 = 2 * precision * recall / (precision + recall)
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
    return precisions, recalls, f1s

def macro_average(values):
    """
    Compute macro-average of a list of values (simple mean).
    """
    if len(values) == 0:
        return 0.0
    return sum(values) / len(values)



- `confusion_matrix`: `mat[true_class][predicted_class]` sayısını tutuyor.
- Her sınıf için `TP`, `FP`, `FN` hesaplanıyor.
- Precision, Recall, F1 her sınıf için ayrı ayrı hesaplanıyor.
- `macro_average` ile sınıflar üzerinden ortalama alıyoruz (macro Precision, macro Recall, macro F1).



## 🧩 Cell 6 – Test set üzerinde metrikleri yazdırma (Code)


In [6]:
# Compute confusion matrix and metrics on test set
cm_test = confusion_matrix(y_test_list, y_test_pred, num_classes)
print("Confusion Matrix (rows = true, cols = pred):")
for row in cm_test:
    print(row)

precisions, recalls, f1s = precision_recall_f1_per_class(
    y_test_list, y_test_pred, num_classes
)
macro_precision = macro_average(precisions)
macro_recall = macro_average(recalls)
macro_f1 = macro_average(f1s)

print("\nPer-class Precision:")
print(precisions)
print("\nPer-class Recall:")
print(recalls)
print("\nPer-class F1:")
print(f1s)
print(f"\nMacro Precision: {macro_precision:.4f}")
print(f"Macro Recall:    {macro_recall:.4f}")
print(f"Macro F1-score:  {macro_f1:.4f}")



Confusion Matrix (rows = true, cols = pred):
[15, 0, 0, 0, 0, 0, 4, 1, 0, 3, 0, 0, 0, 3, 0, 5, 15, 2, 0, 0, 4, 3, 0, 0, 0, 4, 0, 1, 0, 2, 0, 9, 0, 1, 0, 0, 3, 1]
[1, 36, 0, 1, 0, 4, 8, 0, 0, 0, 0, 0, 0, 1, 0, 2, 2, 0, 1, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0]
[3, 0, 0, 0, 0, 3, 1, 0, 0, 3, 1, 0, 0, 0, 0, 13, 7, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0]
[7, 2, 0, 1, 0, 0, 12, 0, 0, 0, 0, 0, 0, 1, 0, 2, 4, 1, 4, 0, 1, 4, 0, 0, 3, 0, 0, 1, 3, 3, 0, 11, 1, 1, 1, 0, 6, 0]
[1, 0, 0, 0, 4, 2, 26, 1, 0, 0, 1, 1, 1, 4, 3, 1, 3, 3, 1, 1, 1, 1, 0, 1, 2, 0, 0, 3, 0, 0, 0, 3, 0, 9, 0, 0, 9, 0]
[0, 3, 0, 0, 0, 24, 3, 3, 0, 4, 4, 0, 0, 0, 0, 11, 24, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0]
[0, 1, 0, 0, 0, 0, 55, 0, 0, 0, 0, 0, 0, 1, 0, 2, 1, 0, 6, 0, 0, 1, 0, 0, 0, 0, 0, 3, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0]
[2, 1, 0, 0, 0, 2, 0, 26, 0, 19, 3, 0, 0, 0, 0, 2, 21, 1, 1, 0, 3, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 1, 2]
[2, 0, 0, 0, 0, 0, 0,

- Test set için confusion matrix tablo şeklinde yazdırılır.
- Her sınıf için Precision / Recall / F1 değerlerini gösterir.
- Macro (sınıflar arası ortalama) değerler de hesaplanır; rapor için kullanışlıdır.

 